# Actividad 5 - Entrenamiento, ajuste y registro con MLflow

Este notebook reproduce la preparacion de datos, entrenamiento de dos modelos, Grid Search con validacion cruzada y registro de resultados en MLflow.

In [ ]:
# 1. Instalacion de dependencias en Colab
!pip install -q pandas numpy scikit-learn matplotlib mlflow

In [ ]:
# 2. Importar librerias
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import mlflow
import mlflow.sklearn

In [ ]:
# 3. Crear estructura del proyecto
BASE_DIR = Path("Actividad5")
for ruta in ["datos/datos_ini", "datos/datos_limp", "fuentes", "figuras", "reportes", "evidencias"]:
    (BASE_DIR / ruta).mkdir(parents=True, exist_ok=True)
print("Estructura creada en", BASE_DIR.resolve())

In [ ]:
# 4. Cargar dataset publico y guardar version original
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
df["target_name"] = df["target"].map({0: "malignant", 1: "benign"})
df.to_csv(BASE_DIR / "datos/datos_ini/breast_cancer_wisconsin_original.csv", index=False)
print(df.shape)
df.head()

In [ ]:
# 5. Limpieza y version limpia
df_limpio = df.drop_duplicates().copy()
for col in data.feature_names:
    df_limpio[col] = pd.to_numeric(df_limpio[col], errors="coerce")
df_limpio = df_limpio.dropna(subset=list(data.feature_names) + ["target"])
df_limpio["target"] = df_limpio["target"].astype(int)
df_limpio["target_name"] = df_limpio["target"].map({0: "malignant", 1: "benign"})
df_limpio.to_csv(BASE_DIR / "datos/datos_limp/breast_cancer_wisconsin_limpio.csv", index=False)
print("Registros originales:", len(df), "Registros limpios:", len(df_limpio))
print("Nulos totales:", int(df_limpio.isna().sum().sum()))

In [ ]:
# 6. Visualizaciones de calidad de datos
ax = df_limpio["target_name"].value_counts().plot(kind="bar")
ax.set_title("Distribucion de clases")
ax.set_xlabel("Clase")
ax.set_ylabel("Numero de registros")
plt.tight_layout()
plt.savefig(BASE_DIR / "figuras/01_distribucion_clases.png", dpi=150)
plt.show()

cols = list(data.feature_names[:10]) + ["target"]
plt.figure(figsize=(8,6))
plt.imshow(df_limpio[cols].corr(), cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(cols)), cols, rotation=90, fontsize=7)
plt.yticks(range(len(cols)), cols, fontsize=7)
plt.title("Correlacion de variables principales")
plt.tight_layout()
plt.savefig(BASE_DIR / "figuras/02_correlacion_variables.png", dpi=150)
plt.show()

In [ ]:
# 7. Configurar MLflow local
mlflow.set_tracking_uri(f"file:{BASE_DIR / 'mlruns'}")
mlflow.set_experiment("Actividad5_BreastCancer_Clasificacion")

In [ ]:
# 8. Preparar entrenamiento
X = df_limpio[list(data.feature_names)]
y = df_limpio["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelos = {
    "Regresion_Logistica": (
        Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=5000, random_state=42))]),
        {"clf__C": [0.01, 0.1, 1, 10], "clf__penalty": ["l2"], "clf__solver": ["lbfgs"]},
    ),
    "Random_Forest": (
        Pipeline([("clf", RandomForestClassifier(random_state=42, class_weight="balanced"))]),
        {"clf__n_estimators": [100, 200], "clf__max_depth": [None, 5, 10], "clf__min_samples_split": [2, 5]},
    ),
}

In [ ]:
# 9. Entrenar, ajustar hiperparametros y registrar en MLflow
resultados = []
for nombre_modelo, (pipeline, grid) in modelos.items():
    with mlflow.start_run(run_name=nombre_modelo):
        busqueda = GridSearchCV(pipeline, grid, scoring="f1", cv=cv, n_jobs=-1, return_train_score=True)
        busqueda.fit(X_train, y_train)
        pred = busqueda.predict(X_test)
        proba = busqueda.predict_proba(X_test)[:, 1]
        metricas = {
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred),
            "recall": recall_score(y_test, pred),
            "f1": f1_score(y_test, pred),
            "roc_auc": roc_auc_score(y_test, proba),
            "cv_f1_mean": busqueda.best_score_,
            "cv_f1_std": busqueda.cv_results_["std_test_score"][busqueda.best_index_],
        }
        mlflow.log_params(busqueda.best_params_)
        for k, v in metricas.items():
            mlflow.log_metric(k, float(v))
        cm = confusion_matrix(y_test, pred)
        plt.figure(figsize=(4,3))
        plt.imshow(cm, cmap="Blues")
        plt.title(f"Matriz de confusion - {nombre_modelo}")
        plt.xlabel("Prediccion")
        plt.ylabel("Real")
        plt.xticks([0,1], ["malignant", "benign"])
        plt.yticks([0,1], ["malignant", "benign"])
        for i in range(2):
            for j in range(2):
                plt.text(j, i, cm[i, j], ha="center", va="center")
        plt.tight_layout()
        cm_path = BASE_DIR / f"figuras/matriz_confusion_{nombre_modelo}.png"
        plt.savefig(cm_path, dpi=150)
        plt.close()
        reporte_path = BASE_DIR / f"reportes/classification_report_{nombre_modelo}.txt"
        reporte_path.write_text(classification_report(y_test, pred, target_names=["malignant", "benign"]), encoding="utf-8")
        mlflow.log_artifact(str(cm_path))
        mlflow.log_artifact(str(reporte_path))
        mlflow.sklearn.log_model(busqueda.best_estimator_, artifact_path=f"modelo_{nombre_modelo}")
        resultados.append({"modelo": nombre_modelo, **metricas, "mejores_parametros": json.dumps(busqueda.best_params_)})

metricas_df = pd.DataFrame(resultados)
metricas_df.to_csv(BASE_DIR / "reportes/metricas_modelos_mlflow.csv", index=False)
metricas_df

In [ ]:
# 10. Grafica comparativa
ax = metricas_df.set_index("modelo")[["accuracy", "precision", "recall", "f1", "roc_auc"]].T.plot(kind="bar", figsize=(8,4.5))
ax.set_title("Comparacion de desempeno por modelo")
ax.set_xlabel("Metrica")
ax.set_ylabel("Valor")
ax.set_ylim(0.85, 1.01)
plt.tight_layout()
plt.savefig(BASE_DIR / "figuras/comparacion_metricas_mlflow.png", dpi=150)
plt.show()

## Abrir MLflow

En Colab puedes ejecutar `!mlflow ui --backend-store-uri Actividad5/mlruns --host 0.0.0.0` y usar una herramienta como ngrok si necesitas exponer la interfaz. Si trabajas localmente, ejecuta en terminal: `mlflow ui --backend-store-uri ./Actividad5/mlruns`.